# CRISPRtracrRNA Prediction

![CRISPRtracrRNA Prediction](https://proto-bio.github.io/proto-assets/images/tool/crispr_tracr_rna/hero.png)

This notebook demonstrates `run_crispr_tracr_rna`, which scans nucleotide sequences for tracrRNA candidates using the CRISPRtracrRNA tool from the Backofen Lab. tracrRNA is required by type II CRISPR-Cas9 systems to form the guide RNA complex; identifying it is a critical step when characterizing novel Cas9 orthologs or validating candidate loci in a CRISPR engineering pipeline.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("crispr_tracr_rna")
display_overview("crispr_tracr_rna")
display_docs_section("crispr_tracr_rna", "Background")

## Available tools

In [ ]:
display_available_tools("crispr_tracr_rna")

### `run_crispr_tracr_rna`

`run_crispr_tracr_rna` predicts tracrRNA sequences from nucleotide sequences that contain CRISPR loci. The tool evaluates anti-repeat complementarity and uses IntaRNA to compute RNA-RNA interaction energies, returning predicted tracrRNA coordinates and interaction energy scores for each input sequence. In practice the input should be a confirmed CRISPR-containing genomic region, typically identified by a tool like MinCED in an earlier analysis step.

In [ ]:
from proto_tools import (
    CrisprTracrRNAInput,
    CrisprTracrRNAConfig,
    run_crispr_tracr_rna,
)

In [ ]:
# Display input docs
display_api_reference("crispr_tracr_rna", "input", "run_crispr_tracr_rna")

from pathlib import Path

# Real Streptococcus pyogenes SF370 CRISPR1-Cas9 locus (NC_002737.2).
locus = "".join(l for l in Path("spyogenes_crispr_locus.fasta").read_text().splitlines() if not l.startswith(">"))
inputs = CrisprTracrRNAInput(sequences=[locus])

In [ ]:
# Display config docs
display_api_reference("crispr_tracr_rna", "config", "run_crispr_tracr_rna")

# Type II model for Cas9-associated tracrRNA prediction | see docs above for all fields
config = CrisprTracrRNAConfig(model_type="II")

In [ ]:
# Run the tracrRNA prediction
result = run_crispr_tracr_rna(inputs, config)

In [ ]:
# Display output docs
display_api_reference("crispr_tracr_rna", "output", "run_crispr_tracr_rna")

# One CrisprTracrRNASequenceResult per input sequence; each carries every candidate hit
# upstream produced for that accession, sorted by score descending.
for seq_result in result.results:
    top = seq_result.top_candidate
    if top is not None and top.has_tracr:
        print(f"{seq_result.sequence_id}: tracrRNA at {top.anti_repeat_start}-{top.anti_repeat_end} "
              f"(score={top.score}, {len(seq_result.candidates)} candidates)")
        print(f"  Interaction energy: {top.interaction_energy} kcal/mol")
    else:
        print(f"{seq_result.sequence_id}: no tracrRNA detected")

## Export Results

TracrRNA predictions can be exported to CSV or JSON. Both formats flatten to one row per candidate hit — sequences with multiple candidates produce multiple rows (sorted by `score` descending), and sequences with no candidates produce a single `sequence_id`-only row.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_tracr_rna_results", export_path=output_dir, file_format="json")
print(f"tracrRNA predictions exported to {output_dir}")